In [27]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils

In [28]:
df_pl = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/pl_bn_no_context.csv")

In [29]:
df_pl = df_pl.drop(["Unnamed: 0"], axis = 1)

In [30]:
df_pl

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,31201.797665,29.28,67.985714,3.630000,2.570000,21.39,6.903,-3.803166,18.166685,-1.872662,5.83,-0.3,17.3,0.000009,62.749213,36.747184
1,33846.381917,28.82,67.985714,3.690000,1.910000,17.10,7.686,-5.237136,18.626276,-3.397579,2.72,0.5,16.9,0.000009,65.250459,57.512070
2,36057.305377,29.96,62.600000,3.740000,2.060000,20.47,8.676,-0.960496,18.707747,-6.157262,3.18,0.8,17.1,0.000008,75.989515,135.142137
3,37895.481599,29.70,62.200000,3.650000,2.230000,23.58,9.281,-2.441439,19.149239,-6.142614,3.94,1.0,17.6,0.000009,84.200841,180.098848
4,41087.789260,30.39,62.500000,3.650000,2.330000,25.53,10.337,-2.287307,18.077861,-4.112383,3.57,0.0,17.7,0.000009,101.918024,247.386988
5,42619.735974,30.23,62.900000,3.810000,2.620000,26.31,10.955,-0.784100,18.008657,-3.437896,1.53,0.0,17.1,0.000008,115.869679,265.323092
6,43053.616233,28.68,63.200000,3.910000,3.050000,27.16,11.452,1.117069,18.330801,-4.023830,0.26,-1.2,17.3,0.000008,111.382755,275.386821
7,44990.974498,29.94,64.900000,4.020000,3.550000,23.73,11.605,0.407324,18.238027,-3.367862,3.85,-0.3,17.0,0.000008,103.683241,307.471216
8,47637.139301,30.99,66.300000,4.070000,3.810000,20.68,11.881,2.067846,17.905031,-2.532527,3.30,-1.0,17.6,0.000008,123.085022,169.995130
9,49142.895037,30.67,68.200000,4.300000,4.000000,17.59,11.396,2.946955,17.821550,-2.650770,3.82,0.2,17.3,0.000008,112.220174,76.266802


In [31]:
pl = pd.DataFrame()
pl = df_pl.copy()
pl['VABI_1'] = pl['VABI'].shift(1)
pl['FDI_3'] = pl['FDI'].shift(3)

In [32]:
pl = pl.dropna()

In [33]:
sm = StructureModel()

In [34]:
sm.add_edges_from([
    ('EUPC', 'REST'),
    ('EUPC', 'ARP'),
    ('EUPC', 'PAM'),
    ('VABI_1', 'EUPC'),
    ('FDI_3', 'PAM')
    
])

In [35]:
sm.edges

OutEdgeView([('EUPC', 'REST'), ('EUPC', 'ARP'), ('EUPC', 'PAM'), ('VABI_1', 'EUPC'), ('FDI_3', 'PAM')])

In [36]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_pl_2007_2015.html")

Graphs/fully_connected_pl_2007_2015.html


In [37]:
bn = BayesianNetwork(sm)

In [40]:
discretised_pl = pd.DataFrame(index=pl.index)

for col in pl.columns:
    no_unique = pl[col].nunique()
    
    if no_unique <= 1:
        discretised_pl[col] = 0
    else:
        try:
            discretised_pl[col] = pd.qcut(
                pl[col], 
                q=min(2, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_pl[col] = pl[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_pl)

Discretised data:
    GDPC  VABI  EMP  QOA  QOR  YUA  REST  EBSG  GGFC  LBGD  FDI  PC  ARP  \
3      0     1    0    0    0    1     0     0     1     0    1   1    1   
4      0     1    0    0    0    1     0     0     0     0    1   1    1   
5      0     1    0    0    0    1     0     0     0     0    0   1    1   
6      0     0    0    0    0    1     0     0     1     0    0   0    1   
7      0     1    0    0    1    1     0     0     1     0    1   0    0   
8      0     1    0    1    1    1     1     1     0     1    0   0    1   
9      0     1    0    1    1    0     0     1     0     1    1   1    1   
10     1     0    1    1    1    0     0     1     0     1    0   1    0   
11     1     0    1    1    1    0     1     0     0     1    0   1    0   
12     1     0    1    1    1    0     1     1     0     1    0   0    0   
13     1     0    1    0    0    0     1     1     1     0    0   0    0   
14     1     0    1    0    0    0     1     1     1     1    1   0   

In [41]:
discretised_pl = discretised_pl.reset_index(drop=True)
bn.fit_node_states(discretised_pl)
baseline_auc = utils.get_avg_auc_all_info(discretised_pl, bn)
print(f"Baseline AUC: {baseline_auc}")

Processing fold 0 using 7 cores takes 5.816804885864258 seconds
Processing fold 1 using 7 cores takes 5.464201927185059 seconds
Processing fold 2 using 7 cores takes 5.5556640625 seconds
Processing fold 3 using 7 cores takes 5.5384509563446045 seconds
Processing fold 4 using 7 cores takes 5.568380117416382 seconds
Baseline AUC: 0.8393518518518519


In [42]:
edges_to_add = [('LV', 'EUPC'), ('LV', 'REST')]
edges_to_remove = [('EUPC', 'REST')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [16]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_pl_1.html")

Graphs/node_added_pl_1.html


In [17]:
discretised_pl['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_pl, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EUPC' and 'REST': {proposed_auc}")

Processing fold 0 using 7 cores takes 6.581211805343628 seconds
Processing fold 1 using 7 cores takes 5.637418985366821 seconds
Processing fold 2 using 7 cores takes 5.719175815582275 seconds
Processing fold 3 using 7 cores takes 5.583175897598267 seconds
Processing fold 4 using 7 cores takes 5.699086666107178 seconds
AUC from adding LV between 'EUPC' and 'REST': 0.8324074074074075


In [18]:
edges_to_add = [('LV', 'EUPC'), ('LV', 'PAM')]
edges_to_remove = [('EUPC', 'PAM')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [19]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_pl_2.html")

Graphs/node_added_pl_2.html


In [20]:
discretised_pl['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_pl, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EUPC' and 'PAM': {proposed_auc}")

Processing fold 0 using 7 cores takes 6.451513767242432 seconds
Processing fold 1 using 7 cores takes 5.4524500370025635 seconds
Processing fold 2 using 7 cores takes 5.674458980560303 seconds
Processing fold 3 using 7 cores takes 5.581997871398926 seconds
Processing fold 4 using 7 cores takes 5.6271398067474365 seconds
AUC from adding LV between 'EUPC' and 'PAM': 0.7944444444444445


In [21]:
edges_to_add = [('LV', 'VABI_1'), ('LV', 'EUPC')]
edges_to_remove = [('VABI_1', 'EUPC')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [22]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_pl_3.html")

Graphs/node_added_pl_3.html


In [23]:
discretised_pl['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_pl, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'VABI_1' and 'EUPC': {proposed_auc}")

Processing fold 0 using 7 cores takes 6.02357816696167 seconds
Processing fold 1 using 7 cores takes 5.384332895278931 seconds
Processing fold 2 using 7 cores takes 5.430954217910767 seconds
Processing fold 3 using 7 cores takes 5.353295803070068 seconds
Processing fold 4 using 7 cores takes 5.689874887466431 seconds
AUC from adding LV between 'VABI_1' and 'EUPC': 0.8592592592592592


In [24]:
edges_to_add = [('LV', 'EUPC'), ('LV', 'ARP')]
edges_to_remove = [('EUPC', 'ARP')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [25]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_pl_4.html")

Graphs/node_added_pl_4.html


In [26]:
discretised_pl['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_pl, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EUPC' and 'ARP': {proposed_auc}")

Processing fold 0 using 7 cores takes 6.045928955078125 seconds
Processing fold 1 using 7 cores takes 5.4370880126953125 seconds
Processing fold 2 using 7 cores takes 5.596078157424927 seconds
Processing fold 3 using 7 cores takes 5.4387428760528564 seconds
Processing fold 4 using 7 cores takes 5.55344820022583 seconds
AUC from adding LV between 'EUPC' and 'ARP': 0.812962962962963
